# Integrating MLflow with DKube for Seamless ML Training

This notebook demonstrates how to leverage an ML training program developed with MLflow within the DKube environment. 

When executed inside the DKube IDE, all MLflow calls are automatically routed to DKube's enterprise-grade MLflow tracking server. This server stores the MLflow records in a database, while the DKube UI offers a multi-user interface for viewing and managing these records.

All MLflow tracking APIs can be used transparently without the need for any DKube-specific code modifications.

In [ ]:
import os
from sklearn.model_selection import train_test_split
from sklearn import preprocessing as skpreprocessing
from sklearn.preprocessing import StandardScaler
import mlflow
from mlflow.models.signature import infer_signature
import pandas as pd

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

import warnings
warnings.filterwarnings("ignore")
import requests, argparse
requests.packages.urllib3.disable_warnings()

### MACROS

In [ ]:
epochs = 10
learning_rate = 0.01
dataset = "https://dkube-examples-data.s3.us-west-2.amazonaws.com/monitoring-insurance/training-data/insurance.csv"
modeldir = os.path.join(os.getcwd(), "mlflow_insurance_model")

os.makedirs(modeldir, exist_ok=True)

### TRAIN-TEST DATASET

In [ ]:
data = pd.read_csv(dataset)
insurance_input = data.drop(['charges','timestamp','unique_id'],axis=1)
insurance_target = data['charges']
    
for col in ['sex', 'smoker', 'region']:
    if (insurance_input[col].dtype == 'object'):
        le = skpreprocessing.LabelEncoder()
        le = le.fit(insurance_input[col])
        insurance_input[col] = le.transform(insurance_input[col])
        print('Completed Label encoding on',col)
    
#standardize data
x_scaled = StandardScaler().fit_transform(insurance_input)
x_train, x_test, y_train, y_test = train_test_split(x_scaled,
                                                    insurance_target.values,
                                                    test_size = 0.25,
                                                    random_state=1211)

### MODEL INITIALIZATION

In [ ]:
tf.random.set_seed(42)  #first we set random seed
model = keras.Sequential([
      layers.InputLayer(input_shape=(6)),
      layers.Dense(64, activation='relu'),
      layers.Dense(64, activation='relu'),
      layers.Dense(1)
  ])

model.compile(loss='mean_absolute_error',
            optimizer=tf.keras.optimizers.Adam(lr=LEARNING_RATE))

### MLFLOW METRICS LOGGING

In [ ]:
# mlflow metric logging
class loggingCallback(keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        mlflow.log_metric("train_loss", logs["loss"], step=epoch)
        mlflow.log_metric("val_loss", logs["val_loss"], step=epoch)
        # output accuracy metric for katib to collect from stdout
        print(f"loss={round(logs['loss'],2)}")

### ML TRAINING with MLFLOW

In [ ]:
import random
import string
run_name = 'mlflow-run-' + ''.join(random.choices(string.ascii_letters, k=6))
run_name = run_name.lower()
with mlflow.start_run(run_name=run_name) as run:
    
    model.fit(x_train, y_train, epochs = NUM_EPOCHS, verbose=0,
                validation_split=0.1, callbacks=[loggingCallback()])
    
    # Exporting model
    model.save(filepath=os.path.join(OUTPUT_MODEL_DIR, '1'))
    
    # Two ways to save model - log_artifacts() or log_model()
    mlflow.log_artifacts(OUTPUT_MODEL_DIR) ## For tf-serving
    signature = infer_signature(x_test, model.predict(x_test))
    mlflow.keras.log_model(keras_model=model, artifact_path=None, signature=signature)
        
    # Record parameters
    mlflow.log_params({"dataset": "https://dkube-examples-data.s3.us-west-2.amazonaws.com/monitoring-insurance/training-data/insurance.csv",
                       "code": "https://github.com/oneconvergence/dkube-examples/tree/training/insurance",
                       "model": "Deep Neural Network"})
    
print(f"Training Complete with dkube run name [{run_name}] ")